# Notebook 03 — Precision Coding

**Duration:** ~30 minutes

In notebook 02 you saw that prompt structure matters. In this notebook we take that further. You will:

1. Build the same request five ways, adding one layer of prompt structure at a time, and see exactly how the output changes.
2. Replicate a published coding method (Tai et al., 2024) by coding a passage from the Privacy Act 2020 for the presence or absence of specific concepts — first by hand, then with the LLM.
3. When you and the LLM disagree, ask the LLM to defend itself, and decide whether its reasoning changes your mind.

The published paper behind this workshop (Ardekani et al., 2026) reports LLM extraction at 86% precision against 96% for rule-based methods. Prompt design is one of the main reasons for that gap. This notebook makes that gap concrete.

## Setup

Run this cell first. If you saved your Groq API key in Colab Secrets during notebook 01, it will load automatically.

In [ ]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

!pip install groq requests lxml Pillow
import os, json, base64, requests, io
from groq import Groq
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

# Load API key from Colab Secrets (set up in notebook 01)
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    os.environ["GROQ_API_KEY"] = "paste_your_key_here"   # <-- fallback: paste key here
    print("Could not load from Secrets. Paste your key in the line above.")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
TEXT_MODEL = "llama-3.3-70b-versatile"
VISION_MODEL = "meta-llama/llama-4-maverick-17b-128e-instruct"

print("Setup complete.")

## The passage we will work with

Today we use a short passage from New Zealand's **Privacy Act 2020**. This Act sets out the **Information Privacy Principles** (IPPs) — 13 rules that agencies must follow when they handle personal information about people.

We will focus on two of those principles:
- **IPP 3** — Collection of information from the person it is about.
- **IPP 11** — Limits on disclosing personal information.

These are substantive rules, written in plain-ish legal English. They give us enough material to summarise, code, and debate.

> Note: the passage below has been adapted for teaching. It follows the structure and wording of the Act but may not be word-for-word with the latest published text. Use the Act itself as the authoritative source for any research work.

Run the cell below to load the passage.

In [ ]:
PRIVACY_ACT_PASSAGE = """
Information Privacy Principle 3 — Collection of information from subject.

Where an agency collects personal information directly from the individual concerned, the agency must take any steps that are, in the circumstances, reasonable to ensure that the individual concerned is aware of:
(a) the fact that the information is being collected;
(b) the purpose for which the information is being collected;
(c) the intended recipients of the information;
(d) the name and address of the agency that is collecting the information and the agency that will hold the information;
(e) whether the supply of the information is voluntary or mandatory, and, if mandatory, the law under which it is required;
(f) the consequences (if any) for the individual if the information is not provided;
(g) the rights of access to, and correction of, that personal information.

Information Privacy Principle 11 — Limits on disclosure of personal information.

An agency that holds personal information must not disclose the information to any other agency or to any person unless the agency believes, on reasonable grounds, that:
(a) the disclosure is one of the purposes in connection with which the information was obtained, or is directly related to those purposes;
(b) the source of the information is a publicly available publication;
(c) the disclosure is authorised by the individual concerned;
(d) non-compliance is necessary to prevent or lessen a serious threat to public health or safety, or to the life or health of any individual.
"""

print(PRIVACY_ACT_PASSAGE)
print(f"Passage length: {len(PRIVACY_ACT_PASSAGE.split())} words")

## Act 1 — Prompt engineering progression

We will summarise the passage above five times. Each version adds one more layer of prompt structure. After all five, we compare them side by side with automatic word counts.

A word on terminology:
- **Prompt** — the text you send to the LLM.
- **Role** — a line that tells the model who it should pretend to be. Sent as the first message of the conversation.
- **Output format** — rules about how the answer should look (length, style, structure).
- **Perspective** — a stance or viewpoint the model should take (e.g. a privacy rights advocate, a business owner).

We will keep `temperature=0` throughout, so any differences come from the prompt, not randomness.

Each answer is saved into a dictionary called `outputs` so we can compare them at the end.

### Version 1 — bare instruction

**Your task:** Write the simplest prompt you can think of — one that asks the model to summarise the passage. No role, no word limit, no formatting rules. Just the bare request.

In the cell below, fill in the `content` field for the user message. You can include the passage inside your prompt by writing `{PRIVACY_ACT_PASSAGE}` somewhere in the string (note the `f` before the opening quote — that is what lets Python substitute the variable in).

In [ ]:
outputs = {}

response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {"role": "user", "content": f""}   # <-- WRITE YOUR PROMPT HERE
    ]
)
outputs["v1 — bare instruction"] = response.choices[0].message.content.strip()
print(outputs["v1 — bare instruction"])

### Version 2 — add a word limit

**Your task:** Write a prompt that asks for a summary of the passage **in under 20 words**. Everything else stays the same as v1 — no role, no formatting rules. We picked a deliberately short limit so we can measure whether the model obeys.

Fill in the `content` field below.

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {"role": "user", "content": f""}   # <-- WRITE YOUR PROMPT HERE
    ]
)
outputs["v2 — + word limit"] = response.choices[0].message.content.strip()
print(outputs["v2 — + word limit"])

### Version 3 — add a role

**Your task:** Keep the word limit from v2, and now add a **role** by writing a system message. Tell the model it is a **policy analyst**. Roles narrow the model's focus by pointing it at a specific professional frame.

Fill in both `content` fields below — the system message (the role) and the user message (the same request as v2).

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {"role": "system", "content": ""},   # <-- WRITE THE ROLE HERE
        {"role": "user", "content": f""}     # <-- WRITE YOUR PROMPT HERE
    ]
)
outputs["v3 — + role"] = response.choices[0].message.content.strip()
print(outputs["v3 — + role"])

### Version 4 — add output format rules

**Your task:** Keep the role and the word limit from v3, and now add **output format rules** to the user message. Ask the model to:

- use **plain English**,
- **avoid legal jargon**,
- return **only the summary** with **no preamble** (so it does not start with phrases like "Here is a summary:").

Fill in both `content` fields below.

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {"role": "system", "content": ""},   # <-- WRITE THE ROLE HERE
        {"role": "user", "content": f""}     # <-- WRITE YOUR PROMPT HERE
    ]
)
outputs["v4 — + output format"] = response.choices[0].message.content.strip()
print(outputs["v4 — + output format"])

### Version 5 — add a perspective (your turn)

This is a fill-in-the-blanks cell. The role now carries a **perspective** — a stance the model should take. The formatting rules are exactly the same as v4.

We suggest starting with: **"a privacy rights advocate concerned with surveillance risks"**. Once you have run it, try other perspectives — a small business owner, a data scientist, a teenager — and see what changes.

In [ ]:
# Fill in the blank below with a perspective. Suggested starting value:
#   "a privacy rights advocate concerned with surveillance risks"
perspective = "a privacy rights advocate concerned with surveillance risks"

v5_system = f"You are {perspective}."
v5_user = (
    "Summarise this section of the Privacy Act in under 20 words. "
    "Use plain English. Avoid legal jargon. Return only the summary. No preamble.\n\n"
    f"{PRIVACY_ACT_PASSAGE}"
)

response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {"role": "system", "content": v5_system},
        {"role": "user", "content": v5_user}
    ]
)
outputs["v5 — + perspective"] = response.choices[0].message.content.strip()
print(outputs["v5 — + perspective"])

### Compare the five versions

The cell below prints a table with the version label, automatic word count, and a trimmed view of each output, followed by the full outputs in order.

In [ ]:
print(f"{'Version':<28} {'Words':>6}   Output (first 60 chars)")
print("-" * 100)
for label, text in outputs.items():
    word_count = len(text.split())
    first_line = text.replace("\n", " ")
    if len(first_line) > 60:
        first_line = first_line[:57] + "..."
    print(f"{label:<28} {word_count:>6}   {first_line}")

print()
print("Full outputs:")
print("=" * 60)
for label, text in outputs.items():
    print(f"\n[{label}]  ({len(text.split())} words)")
    print(text)

### Discussion

Look at the table above and answer these for yourself:

1. **Which version would you cite in a methods section?** A vague prompt makes your method unreproducible. A prompt with role, format, and perspective is explicit — another researcher could run the same prompt and get a near-identical result.
2. **Which version would give the most consistent results if you ran it again?** The more structure a prompt has, the smaller the space the model has to wander in. v4 and v5 will almost always beat v1 on consistency.
3. **Why does v5 emphasise different things from v4, even though the format rules are identical?** A perspective is not neutral. Telling the model to take the viewpoint of a privacy rights advocate changes which parts of the passage it treats as important. This is not a bug — it is the whole point. You now have a lever to shift what the model sees.

We return to this lever in notebook 04, where we run the same analysis from different perspectives to stress-test a set of themes.

## Act 2 — Presence / absence coding (Tai et al. replication)

> Tai et al. (2024) used an LLM to code whether specific psychological constructs — words like persistence, admiration, frustration — were present or absent in interview transcripts. They found the LLM agreed with human coders at a rate comparable to inter-rater reliability between two humans. We are going to replicate that logic using the Privacy Act 2020.

**Presence/absence coding** is one of the simplest forms of content coding. For each concept in your coding scheme, you mark `1` if it is present in the text (directly stated or clearly implied) and `0` if it is absent. It is a standard technique in qualitative and mixed-methods research.

We will code the passage above for these five concepts:

- **consent** — did the individual agree to what is happening to their information?
- **individual** — is a specific person the subject of the rule?
- **purpose** — why is the information being collected or disclosed?
- **disclosure** — is information being shared with others?
- **collect** — is information being gathered?

You will do this **twice**: first by yourself, then with the LLM. Then we compare.

### Step 1 — Your coding

Read the passage one more time (it is printed below as a reminder). For each keyword, decide: is the concept **present** (directly stated or clearly implied) or **absent**?

Then edit the `your_coding` dictionary in the cell — replace each `None` with `1` for present or `0` for absent. **Do this before running the LLM cell** so you are not biased by the model's answer.

In [ ]:
print(PRIVACY_ACT_PASSAGE)
print("=" * 60)

KEYWORDS = ["consent", "individual", "purpose", "disclosure", "collect"]

# Read the passage above. For each keyword, enter 1 if the concept is present
# (directly stated or clearly implied) or 0 if absent. Do this BEFORE running
# the next cell.
your_coding = {
    "consent": None,
    "individual": None,
    "purpose": None,
    "disclosure": None,
    "collect": None,
}

# Sanity check — reminds you if any values are still None
if any(v is None for v in your_coding.values()):
    print("Warning: some keywords still have value None. Fill them in before running the next cell.")
else:
    print("Your coding is complete:")
    for k, v in your_coding.items():
        print(f"  {k:<12} -> {v}")

### Step 2 — LLM coding

Now we ask the LLM the same question. We use a strict prompt: return only a JSON object, nothing else. That makes the response easy to parse programmatically.

JSON stands for **JavaScript Object Notation**. It is a simple text format for structured data — a set of key–value pairs in curly braces, like `{"consent": 1, "individual": 0}`.

In [ ]:
coding_system = (
    "You are a qualitative researcher performing presence/absence coding on legislative text. "
    "For each keyword, decide whether the concept is present (directly stated or clearly implied) or absent. "
    "Return ONLY a JSON object mapping each keyword to 1 (present) or 0 (absent). "
    "No explanation. No preamble. No code fences."
)

coding_user = (
    f"Text:\n{PRIVACY_ACT_PASSAGE}\n\n"
    f"Keywords to code: {KEYWORDS}\n\n"
    "Return a JSON object with each keyword as a key and 1 or 0 as the value."
)

response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {"role": "system", "content": coding_system},
        {"role": "user", "content": coding_user}
    ]
)

raw_reply = response.choices[0].message.content.strip()
print("LLM raw reply:")
print(raw_reply)
print()

# Parse JSON. Strip markdown code fences if the model added them.
def parse_json_reply(text):
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        lines = [l for l in lines if not l.startswith("```")]
        text = "\n".join(lines).strip()
    return json.loads(text)

try:
    parsed = parse_json_reply(raw_reply)
    # Ensure all five keywords are present; anything missing defaults to 0
    llm_coding = {k: int(parsed.get(k, 0)) for k in KEYWORDS}
    print("Parsed LLM coding:")
    for k, v in llm_coding.items():
        print(f"  {k:<12} -> {v}")
except (json.JSONDecodeError, ValueError, TypeError) as err:
    print(f"Could not parse LLM output as JSON. Error: {err}")
    print("Falling back to zeros. Re-run the cell or inspect the raw reply above.")
    llm_coding = {k: 0 for k in KEYWORDS}

### Step 3 — Comparison

Put the two codings side by side and count the agreements.

In [ ]:
print(f"{'Keyword':<12} {'Your coding':<14} {'LLM coding':<14} Agreement")
print("-" * 55)

agreements = 0
for k in KEYWORDS:
    your_val = your_coding[k]
    llm_val = llm_coding.get(k)
    agree = your_val == llm_val
    if agree:
        agreements += 1
    mark = "agree" if agree else "DISAGREE"
    your_display = str(your_val) if your_val is not None else "(not coded)"
    print(f"{k:<12} {your_display:<14} {str(llm_val):<14} {mark}")

print()
print(f"You agreed with the LLM on {agreements} out of {len(KEYWORDS)} keywords.")

## Act 3 — Disagreement resolution

When you and the LLM disagree, one of you is wrong — or you are reading the text through different lenses. Either way, the disagreement is useful information. This is the punchline of the whole exercise: **disagreement is not failure, it is evidence about how well the method is working**.

We will now find every disagreement, ask the LLM to explain its reasoning in 2–3 sentences, and let you decide whether the explanation changes your mind.

In [ ]:
disagreements = [k for k in KEYWORDS if your_coding[k] != llm_coding.get(k)]

if not disagreements:
    print("No disagreements — you and the LLM coded all five keywords the same way. Nicely done.")
    print()
    print("To try out the resolution step anyway:")
    print("  1. Go back to the `your_coding` cell.")
    print("  2. Deliberately flip one of your values (e.g. change a 1 to a 0).")
    print("  3. Re-run the LLM coding cell, the comparison cell, and this cell.")
else:
    print(f"Found {len(disagreements)} disagreement(s). Asking the LLM to explain each one.")
    print()

    for k in disagreements:
        your_val = your_coding[k]
        llm_val = llm_coding.get(k)
        print("=" * 60)
        print(f"Keyword: {k}")
        print(f"Your coding: {your_val}    LLM coding: {llm_val}")
        print("-" * 60)

        explain_system = (
            "You are a qualitative researcher explaining a presence/absence coding decision."
        )
        llm_label = "present" if llm_val == 1 else "absent"
        explain_user = (
            f"You coded the keyword '{k}' as {llm_val} ({llm_label}) in the following text.\n\n"
            f"Text:\n{PRIVACY_ACT_PASSAGE}\n\n"
            "Explain your reasoning in 2-3 sentences. Be specific about which "
            "part of the text led to your decision. If the concept is implied "
            "rather than stated, say so and point to the phrase you took as an implication."
        )

        response = client.chat.completions.create(
            model=TEXT_MODEL,
            temperature=0,
            messages=[
                {"role": "system", "content": explain_system},
                {"role": "user", "content": explain_user}
            ]
        )
        print(response.choices[0].message.content.strip())
        print()
        print("Do you agree with this reasoning? Note your answer — this is your spot-check record.")
        print()

### Reflection

When you disagreed with the LLM and asked for its reasoning — did the explanation change your mind? If yes, the LLM surfaced something you missed. If no, the LLM made an error your manual check caught. Both outcomes are valid data about the reliability of this method. Tai et al. (2024) found LLMs agreed with human coders approximately 86% of the time on presence/absence coding tasks. How does your agreement rate today compare? What would you do differently if you were coding 500 sections rather than one?

## What you accomplished

In this notebook you:

- Saw five versions of the same prompt and measured exactly how each layer of structure (word limit, role, output format, perspective) changes the output.
- Replicated the presence/absence coding method from Tai et al. (2024) on a passage of real NZ legislation.
- Compared your own coding to the LLM's, counted agreements, and asked the LLM to defend each disagreement.

The key insight: a prompt is not a casual question, it is a **method**. The clearer and more specific you make it, the more reproducible your results become — and the more honestly you can describe what you did in a methods section.

**Next up:** In this notebook you controlled the model's output through prompt structure, and you tested its coding reliability against your own judgement. In the next notebook we use those same skills — structured prompts, perspective shifts, and manual verification — to do something more ambitious: generating and validating themes across the entire Act.